## Silver Compras

In [0]:
from pyspark.sql.functions import concat_ws, col, lit, current_timestamp, when, upper, trim, to_date, coalesce, split, datediff, date_format, row_number, initcap, lpad, length, concat
from pyspark.sql.window import Window
from datetime import datetime
from dateutil.relativedelta import relativedelta

df_compras_importar = spark.table('azdb_sales_store.linio.bronze_compras')

df_compras_clean = (
    df_compras_importar
    .select(
        col("venta_id").cast("int"),
        upper(trim(col("factura"))).alias("factura"),
        col("fecha_orden").cast("date"),
        to_date(col("fecha_entrega"), "dd/MM/yyyy").alias("fecha_entrega"),
        to_date(col("fecha_envio"), "dd-MM-yy").alias("fecha_envio"),
        col("estado").cast("int"),
        col("cliente_code"),
        col("tipo_cliente"),
        col("nombres"),
        col("apellidos"),
        coalesce(trim(col("vendedor")), lit("No Identificado")).alias("vendedor"),
        trim(col("departamento")).alias("departamento"),
        trim(col("metodo_pago")).alias("metodo_pago"),
        col("tipo_compra"),
        col("fecha_carga")
    )
)


df_compras_transform_pre = (
    df_compras_clean
    .select(
        col("venta_id"),
        col("factura"),
        col("fecha_orden"),
        col("fecha_entrega"),
        col("fecha_envio"),
        col("estado"),
        when(col("estado") == 5, datediff(col("fecha_envio"), col("fecha_orden"))).otherwise(lit(None)).alias("dias_envio"),
        col("cliente_code"),
        trim(split(col("cliente_code"), "-")[0]).cast("int").alias("cliente_id"),
        trim(split(col("cliente_code"), "-")[1]).alias("num_documento"),
        col("tipo_cliente"),
        col("nombres"),
        col("apellidos"),
        col("vendedor"),
        col("departamento"),
        col("metodo_pago"),
        col("tipo_compra"),
        col("fecha_carga")
    )
)


df_compras_transform = (
    df_compras_transform_pre
    .select(
        col("venta_id"),
        col("factura"),
        col("fecha_orden"),
        col("fecha_entrega"),
        col("fecha_envio"),
        col("estado"),
        col("dias_envio"),
        when(col("dias_envio").isNull(), None)
        .when(col("dias_envio") <= 3, "[0 - 3 días]")
        .when(col("dias_envio") <= 7, "[4 - 7 días]")
        .otherwise("[más de 8 días]")
        .alias("grupo_dias_envio"),
        col("cliente_code"),
        col("cliente_id"),
        col("num_documento"),
        col("tipo_cliente"),
        col("nombres"),
        col("apellidos"),
        col("vendedor"),
        col("departamento"),
        col("metodo_pago"),
        col("tipo_compra"),
        col("fecha_carga"),
    )
)


##########
df_compras = (
    df_compras_transform
    .select(
        col("venta_id"),
        col("factura"),
        col("tipo_compra"),
        col("fecha_orden"),
        col("fecha_entrega"),
        col("fecha_envio"),
        col("estado"),
        col("cliente_id"),
        col("vendedor"),
        col("departamento"),
        col("metodo_pago"),
        col("dias_envio"),
        col("grupo_dias_envio"),
        to_date(date_format(col("fecha_orden"), "yyyy-MM-01")).alias("periodo"),
        current_timestamp().alias("fecha_actualizacion")
    )
)

df_compras.write.format("delta").mode("overwrite").partitionBy("periodo").saveAsTable("azdb_sales_store.linio.silver_compras")



## Silver Cliente

In [0]:
df_clientes_transform = (
    df_compras_transform
    .select(
        col("cliente_id"),
        initcap(trim(col("nombres"))).alias("nombres"),
        initcap(trim(col("apellidos"))).alias("apellidos"),
        upper(trim(col("tipo_cliente"))).alias("tipo_cliente"),
        when(length(col("num_documento")) < 8,lpad(col("num_documento"), 8, "0")).otherwise(col("num_documento")).alias("num_documento"),
        row_number().over(
            Window.partitionBy(col("cliente_id")).orderBy(col("fecha_orden").desc())
        ).alias("rn")
    )
    .filter(col("rn") == 1)
    .drop("rn")
)

df_clientes_transform_1 = (
    df_clientes_transform
    .select(
        col("*"),
        when(length(col("num_documento")) == 8, "DNI")
        .when((length(col("num_documento")) == 11) & col("num_documento").startswith("10"), "RUC10")
        .when((length(col("num_documento")) == 11) & col("num_documento").startswith("20"), "RUC20")
        .otherwise(None)
        .alias("tipo_documento"),
        concat_ws(" ", col("nombres"), col("apellidos")).alias("nombre_completo"),
        current_timestamp().alias("fecha_actualizacion")
    )
)



df_clientes = (
    df_clientes_transform_1
    .select(
        "cliente_id","tipo_documento","num_documento","nombre_completo","tipo_cliente","fecha_actualizacion"
    )
)

df_clientes.write.format("delta").mode("overwrite").saveAsTable("azdb_sales_store.linio.silver_clientes")


## Silver Productos

In [0]:
from pyspark.sql.functions import concat_ws, col, lit, current_timestamp, when, upper, trim, to_date, coalesce, split, datediff, date_format, row_number, initcap, lpad, length, concat
from pyspark.sql.window import Window

df_detalles_import = spark.table('azdb_sales_store.linio.bronze_detalles')

df_detalles_transform = (
    df_detalles_import
    .select(
        col("subcategoria"),
        col("categoria"),
        col("producto_id"),
        trim(col("producto")).alias("producto"),
        col("detalle_id").cast("int").alias("detalle_id"),
        col("unidades").cast("int").alias("unidades"),
        col("oferta_id").cast("int").alias("oferta_id"),
        col("precio_unitario").cast("double").alias("precio_unitario"),
        upper(trim(col("factura"))).alias("factura"),
        (col("unidades").cast("int") * col("precio_unitario").cast("double")).alias("subtotal"), #Hacer el round es opcional
        split(col("nombre_archivo"), r"\\").getItem(0).alias("tienda")
    )
)

df_productos = (
    df_detalles_transform
    .dropDuplicates(["producto"])
    .select(
        col("producto"),
        upper(trim(col("subcategoria"))).alias("subcategoria"),
        upper(trim(col("categoria"))).alias("categoria"),
        current_timestamp().alias("fecha_actualizacion")
    )
)

df_productos.write.format("delta").mode("overwrite").saveAsTable("azdb_sales_store.linio.silver_productos")


## Silver Detalles Prod

In [0]:
df_productos_new = spark.table("azdb_sales_store.linio.silver_productos").select("producto_id","producto")

df_detalles = (
    df_detalles_transform.alias("a")
    .join(df_productos_new.alias("b"), "producto", "inner")
    .select(
        col("a.detalle_id"),
        col("a.factura"),
        col("a.producto"),
        col("b.producto_id").cast("int").alias("producto_id"),  # solo el producto_id de b
        col("a.unidades"),
        col("a.precio_unitario"),
        col("a.subtotal"),
        current_timestamp().alias("fecha_actualizacion")
    )
)


df_detalles.write.format("delta").mode("overwrite").saveAsTable("azdb_sales_store.linio.silver_detalles")
